# Free Llama LoRA fine-tuning with Kaggle or Colab

Use this when Hugging Face Spaces asks for prepaid credits. Upload this notebook and `02_train_huggingface_chat_utf8.jsonl` to the same working directory. Start with Llama 3.2 1B first.


In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets transformers safetensors


In [ ]:
import os
import getpass
from pathlib import Path

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste Hugging Face write token: ")

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_REPO = "Phuoc20050911/copypro-brand-voice-llama-1b-lora"
DATA_PATH = Path("02_train_huggingface_chat_utf8.jsonl")

assert DATA_PATH.exists(), f"Missing {DATA_PATH}. Upload it next to this notebook."
print("Ready", BASE_MODEL, "->", OUTPUT_REPO)


In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")

def format_item(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_item, remove_columns=dataset.column_names)
print(dataset[0]["text"][:500])


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        output_dir="copypro_llama_lora_output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)

trainer.train()


In [ ]:
model.push_to_hub(OUTPUT_REPO, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(OUTPUT_REPO, token=os.environ["HF_TOKEN"])
print("Pushed LoRA adapter to", OUTPUT_REPO)
